In [ ]:
!wget ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
!tar -xzf sift.tar.gz

--2026-05-24 10:43:51--  ftp://ftp.irisa.fr/local/texmex/corpus/sift.tar.gz
           => ‘sift.tar.gz’
Resolving ftp.irisa.fr (ftp.irisa.fr)... 131.254.254.45, 2001:660:7303:254::45
Connecting to ftp.irisa.fr (ftp.irisa.fr)|131.254.254.45|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /local/texmex/corpus ... done.
==> SIZE sift.tar.gz ... 168280445
==> PASV ... done.    ==> RETR sift.tar.gz ... done.
Length: 168280445 (160M) (unauthoritative)

sift.tar.gz         100%[===================>] 160.48M  24.6MB/s    in 7.4s    

2026-05-24 10:44:00 (21.6 MB/s) - ‘sift.tar.gz’ saved [168280445]



In [ ]:
# ============================================================
# GRAND FINALE: LSH στο Πλήρες Dataset (SIFT1M)
# ============================================================

import numpy as np
import time
from pyspark.sql import SparkSession

print("="*50)
while True:
    try:
        user_input = input("Παρακαλώ εισάγετε το κατώφλι απόστασης Epsilon (π.χ. 200.0): ")
        EPSILON = float(user_input)
        break
    except ValueError:
        print("Σφάλμα: Παρακαλώ εισάγετε έναν έγκυρο αριθμό.")
print("="*50)
L_TABLES = 50
K_FUNCS  = 4
W_FACTOR = 1.5
W_WIDTH  = EPSILON * W_FACTOR

CHUNK_SIZE = 1000
D_DIM    = 128
REC_LEN  = 4 + D_DIM * 4
R_PATH   = "sift/sift_base.fvecs"
S_PATH   = "sift/sift_query.fvecs"

# ==== 2. Spark Session ====
spark = SparkSession.builder \
    .appName("LSH_Grand_Finale") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.python.worker.memory", "2g") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "1g") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print(f"Ξεκινάει το Grand Finale στο Spark {spark.version}...")

# ==== 3. Συναρτήσεις (Ίδιες με πριν) ====
def parse_fvecs(record):
    return np.frombuffer(bytes(record[4:]), dtype=np.float32).copy()

def generate_lsh_model(d, L, K, w, seed=42):
    np.random.seed(seed)
    A = [np.random.randn(K, d).astype(np.float32) for _ in range(L)]
    B = [np.random.uniform(0, w, K).astype(np.float32) for _ in range(L)]
    return A, B

def build_bucket_index(S_dict, A, B, w, L):
    buckets = {}
    for s_id, s_vec in S_dict.items():
        for l in range(L):
            proj = np.dot(A[l], s_vec) + B[l]
            key  = (l, tuple(np.floor(proj / w).astype(np.int32).tolist()))
            buckets.setdefault(key, []).append(s_id)
    return buckets

def _process_chunk(ids, vecs, A, B, w, L, s_bkts, s_vecs, eps_sq):
    M = np.stack(vecs)
    candidates = [set() for _ in range(len(ids))]
    for l in range(L):
        projs   = M @ A[l].T + B[l]
        buckets = np.floor(projs / w).astype(np.int32)
        for i in range(len(ids)):
            key = (l, tuple(buckets[i].tolist()))
            if key in s_bkts:
                candidates[i].update(s_bkts[key])
    for i, r_id in enumerate(ids):
        r = M[i]
        for s_id in candidates[i]:
            diff = r - s_vecs[s_id]
            if float(np.dot(diff, diff)) <= eps_sq:
                yield (r_id, s_id)

def lsh_map_chunked(iterator, A_bc, B_bc, w, L, Sb_bc, Sv_bc, eps):
    A = A_bc.value; B = B_bc.value
    sb = Sb_bc.value; sv = Sv_bc.value
    eps_sq = eps ** 2
    ids, vecs = [], []
    for r_id, r_vec in iterator:
        ids.append(r_id); vecs.append(r_vec)
        if len(ids) >= CHUNK_SIZE:
            yield from _process_chunk(ids, vecs, A, B, w, L, sb, sv, eps_sq)
            ids, vecs = [], []
    if ids:
        yield from _process_chunk(ids, vecs, A, B, w, L, sb, sv, eps_sq)

# ==== 4. Φόρτωση ΠΛΗΡΟΥΣ Dataset (Χωρίς Φίλτρα) ====
print("Φόρτωση ολόκληρου του SIFT1M (1M Base x 10K Query)...")
N_PARTS = max(sc.defaultParallelism * 4, 20)

# Προσοχή: Εδώ πλέον ΔΕΝ κάνουμε .collect() το R στη μνήμη του Driver!
rdd_R_kv = (sc.binaryRecords(R_PATH, REC_LEN)
              .map(parse_fvecs).zipWithIndex()
              .map(lambda x: (f"R_{x[1]}", x[0]))
              .repartition(N_PARTS)) # Χωρίς cache για να μην σκάσει η RAM

S_dict = dict(sc.binaryRecords(S_PATH, REC_LEN)
                .map(parse_fvecs).zipWithIndex()
                .map(lambda x: (f"S_{x[1]}", x[0]))
                .collect())

print(f"  Φορτώθηκαν {len(S_dict):,} ερωτήματα S.")

# ==== 5. Εκτέλεση LSH ====
print(f"\nΔημιουργία LSH Μοντέλου (L={L_TABLES}, w={W_WIDTH})...")
A_mat, B_mat = generate_lsh_model(D_DIM, L_TABLES, K_FUNCS, W_WIDTH)
sb_local     = build_bucket_index(S_dict, A_mat, B_mat, W_WIDTH, L_TABLES)

A_bc  = sc.broadcast(A_mat)
B_bc  = sc.broadcast(B_mat)
Sv_bc = sc.broadcast(S_dict)
Sb_bc = sc.broadcast(sb_local)

print("Ξεκινάει ο κατανεμημένος υπολογισμός σε όλο το Cluster...")
t0 = time.time()
_L, _w, _e = L_TABLES, W_WIDTH, EPSILON
# Επειδή το αποτέλεσμα μπορεί να είναι τεράστιο, απλά μετράμε το πλήθος (count)
total_matches = (rdd_R_kv
            .mapPartitions(lambda it,
                _A=A_bc, _B=B_bc, _Sb=Sb_bc, _Sv=Sv_bc,
                _l=_L, _ww=_w, _ee=_e:
                lsh_map_chunked(it, _A, _B, _ww, _l, _Sb, _Sv, _ee))
            .count())
t_lsh = time.time() - t0

print(f"\n{'='*50}")
print(f"🎉 GRAND FINALE ΟΛΟΚΛΗΡΩΘΗΚΕ!")
print(f"  Συνολικά Ζευγάρια που βρέθηκαν: {total_matches:,}")
print(f"  Συνολικός Χρόνος Εκτέλεσης: {t_lsh:.2f} δευτερόλεπτα ({(t_lsh/60):.2f} λεπτά)")
print(f"{'='*50}")

A_bc.destroy(); B_bc.destroy(); Sv_bc.destroy(); Sb_bc.destroy()
sc.stop()

🚀 Ξεκινάει το Grand Finale στο Spark 4.0.2...
Φόρτωση ολόκληρου του SIFT1M (1M Base x 10K Query)...
  Φορτώθηκαν 10,000 ερωτήματα S.

Δημιουργία LSH Μοντέλου (L=50, w=300.0)...
Ξεκινάει ο κατανεμημένος υπολογισμός σε όλο το Cluster...

🎉 GRAND FINALE ΟΛΟΚΛΗΡΩΘΗΚΕ!
  Συνολικά Ζευγάρια που βρέθηκαν: 1,767,676
  Συνολικός Χρόνος Εκτέλεσης: 4277.97 δευτερόλεπτα (71.30 λεπτά)
